## LLMs auf CIC-IDS2017
 
Ich werde sogar lokale als auch cloud Modelle testen. Alle mit und ohnr Beispiel in Prompt.

In [21]:
import requests
import time

ollama_url = 'http://localhost:11434'
model = 'llama3.1:8b'


In [22]:
anfrage = {
    'model': model,
    'messages': [
        {'role': 'user', 'content': 'What is a DoS attack? Answer in one sentence.'}
    ],
    'stream': False,
}

antwort = requests.post(f'{ollama_url}/api/chat', json=anfrage)
daten = antwort.json()
print(daten['message']['content'])

A Denial of Service (DoS) attack is a type of cyberattack where an attacker floods a computer or network with traffic in an attempt to overwhelm its resources and make it unavailable to users.


In [23]:
for i in range(3):
    t0 = time.perf_counter()
    requests.post(f'{ollama_url}/api/chat', json=anfrage)
    dauer = time.perf_counter() - t0
    print(f'Aufruf {i + 1}: {dauer:.2f} s')

Aufruf 1: 1.00 s
Aufruf 2: 0.92 s
Aufruf 3: 0.91 s


### 2. Daten laden
Die LLMs bekommen dieselbe Stichprobe von 200 Flows wie der Random Forest (aus Notebook 02).

In [24]:
import pandas as pd
from pathlib import Path

ergebnis_ordner = Path('results')

stichprobe = pd.read_csv(ergebnis_ordner / 'llm_stichprobe_200.csv', index_col='Original_Index')
print('Zeilen, Spalten:', stichprobe.shape)
stichprobe['Label'].value_counts()

Zeilen, Spalten: (200, 78)


Label
BENIGN              100
DoS Hulk             35
DoS GoldenEye        21
DoS slowloris        21
DoS Slowhttptest     20
Heartbleed            3
Name: count, dtype: int64

In [25]:
importance = pd.read_csv(ergebnis_ordner / 'rf_feature_importance.csv', index_col=0)
prompt_features = importance.index[:20].tolist()
prompt_features

['Packet Length Variance',
 'Packet Length Std',
 'Bwd Packet Length Std',
 'Bwd Packet Length Mean',
 'Average Packet Size',
 'Bwd Packet Length Max',
 'Packet Length Max',
 'Subflow Bwd Bytes',
 'Packet Length Mean',
 'Bwd Segment Size Avg',
 'Total Length of Bwd Packet',
 'RST Flag Count',
 'FWD Init Win Bytes',
 'Fwd Packet Length Max',
 'Bwd Init Win Bytes',
 'Fwd Header Length',
 'Bwd PSH Flags',
 'ACK Flag Count',
 'Total Bwd packets',
 'Bwd Header Length']

In [26]:
def flow_als_text(zeile):
    zeilen = []
    for feature in prompt_features:
        wert = zeile[feature]
        zeilen.append(f'- {feature}: {wert:.6g}')
    return '\n'.join(zeilen)


erster_flow = stichprobe.iloc[0]
print(flow_als_text(erster_flow))
print()
print('Wahres Label:', erster_flow['Label'])

- Packet Length Variance: 85.3333
- Packet Length Std: 9.2376
- Bwd Packet Length Std: 0
- Bwd Packet Length Mean: 60
- Average Packet Size: 52
- Bwd Packet Length Max: 60
- Packet Length Max: 60
- Subflow Bwd Bytes: 30
- Packet Length Mean: 52
- Bwd Segment Size Avg: 60
- Total Length of Bwd Packet: 120
- RST Flag Count: 0
- FWD Init Win Bytes: 0
- Fwd Packet Length Max: 44
- Bwd Init Win Bytes: 0
- Fwd Header Length: 16
- Bwd PSH Flags: 0
- ACK Flag Count: 0
- Total Bwd packets: 2
- Bwd Header Length: 16

Wahres Label: BENIGN


### 3. Prompt
Der Prompt ist auf Englisch und nennt weder den Datensatz noch die Angriffsarten,
damit das Modell nicht auf auswendig gelerntes Wissen über CIC-IDS2017 gelenkt wird.
Die Antwort ist ein festes JSON-Format mit Label und kurzer Begründung.

In [27]:
system_prompt = (
    'You are a security analyst in a Security Operations Center (SOC). '
    'You receive statistics of a single bidirectional network flow. '
    'Sizes are in bytes. '
    'Decide whether the flow is part of an attack or benign traffic.\n'
    'Answer ONLY with a JSON object of the form '
    '{"label": "ATTACK" or "BENIGN", "reason": "<one short sentence>"}'
)


def user_prompt(zeile):
    return 'Flow to classify:\n' + flow_als_text(zeile)


print(system_prompt)
print()
print(user_prompt(erster_flow))

You are a security analyst in a Security Operations Center (SOC). You receive statistics of a single bidirectional network flow. Sizes are in bytes. Decide whether the flow is part of an attack or benign traffic.
Answer ONLY with a JSON object of the form {"label": "ATTACK" or "BENIGN", "reason": "<one short sentence>"}

Flow to classify:
- Packet Length Variance: 85.3333
- Packet Length Std: 9.2376
- Bwd Packet Length Std: 0
- Bwd Packet Length Mean: 60
- Average Packet Size: 52
- Bwd Packet Length Max: 60
- Packet Length Max: 60
- Subflow Bwd Bytes: 30
- Packet Length Mean: 52
- Bwd Segment Size Avg: 60
- Total Length of Bwd Packet: 120
- RST Flag Count: 0
- FWD Init Win Bytes: 0
- Fwd Packet Length Max: 44
- Bwd Init Win Bytes: 0
- Fwd Header Length: 16
- Bwd PSH Flags: 0
- ACK Flag Count: 0
- Total Bwd packets: 2
- Bwd Header Length: 16


In [28]:
def frage_llm(system, user):
    anfrage = {
        'model': model,
        'messages': [
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': user},
        ],
        'stream': False,
        'format': 'json',
        'options': {'temperature': 0, 'seed': 42, 'num_ctx': 8192},
    }
    t0 = time.perf_counter()
    antwort = requests.post(f'{ollama_url}/api/chat', json=anfrage, timeout=600)
    dauer_ms = 1000 * (time.perf_counter() - t0)
    daten = antwort.json()
    return daten['message']['content'], dauer_ms, daten['prompt_eval_count'], daten['eval_count']

In [29]:
text, ms, tokens_in, tokens_out = frage_llm(system_prompt, user_prompt(erster_flow))

print('Antwort des Modells:', text)
print(f'Dauer: {ms:.0f} ms | Tokens: {tokens_in} gelesen, {tokens_out} geschrieben')
print('Wahres Label:', erster_flow['Label'])

Antwort des Modells: {
  "label": "ATTACK",
  "reason": "High packet length variance and standard deviation indicate potential malicious activity"
}
Dauer: 4300 ms | Tokens: 283 gelesen, 28 geschrieben
Wahres Label: BENIGN


In [30]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score


def metriken(df):
    y_wahr = df['y']
    y_llm = (df['label_llm'] != 'BENIGN').astype(int)   # UNKLAR zählt als Angriff
    tn, fp, fn, tp = confusion_matrix(y_wahr, y_llm, labels=[0, 1]).ravel()
    return {
        'FPR': fp / (fp + tn),
        'Recall': recall_score(y_wahr, y_llm, zero_division=0),
        'Precision': precision_score(y_wahr, y_llm, zero_division=0),
        'F1': f1_score(y_wahr, y_llm, zero_division=0),
        'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp,
    }

In [31]:
import json


def lies_label(text):
    try:
        label = json.loads(text)['label'].upper()
    except (json.JSONDecodeError, KeyError, AttributeError):
        return 'UNKLAR'
    if label in ('ATTACK', 'BENIGN'):
        return label
    return 'UNKLAR'


lies_label(text)

'ATTACK'

In [32]:
n_test = 20
ergebnisse = []

frage_llm(system_prompt, user_prompt(stichprobe.iloc[0]))   # Aufwärmen, wird nicht gemessen

for flow_id, zeile in stichprobe.head(n_test).iterrows():
    text, ms, t_in, t_out = frage_llm(system_prompt, user_prompt(zeile))
    label_llm = lies_label(text)
    ergebnisse.append({
        'flow_id': flow_id,
        'label_wahr': zeile['Label'],
        'y': zeile['y'],
        'antwort': text,
        'label_llm': label_llm,
        'latenz_ms': ms,
        'tokens_in': t_in,
        'tokens_out': t_out,
    })
    print(f"{flow_id}: wahr = {zeile['Label']:18s} LLM = {label_llm:7s} ({ms:.0f} ms)")

test = pd.DataFrame(ergebnisse)

388132: wahr = BENIGN             LLM = ATTACK  (635 ms)
485431: wahr = BENIGN             LLM = ATTACK  (721 ms)
197415: wahr = BENIGN             LLM = ATTACK  (726 ms)
495186: wahr = DoS GoldenEye      LLM = ATTACK  (722 ms)
26530: wahr = DoS slowloris      LLM = ATTACK  (743 ms)
104972: wahr = DoS Hulk           LLM = ATTACK  (721 ms)
471952: wahr = BENIGN             LLM = ATTACK  (715 ms)
470842: wahr = BENIGN             LLM = BENIGN  (762 ms)
13817: wahr = BENIGN             LLM = ATTACK  (731 ms)
11984: wahr = DoS slowloris      LLM = BENIGN  (658 ms)
13320: wahr = DoS slowloris      LLM = BENIGN  (573 ms)
185749: wahr = BENIGN             LLM = ATTACK  (716 ms)
237195: wahr = BENIGN             LLM = ATTACK  (728 ms)
438286: wahr = BENIGN             LLM = ATTACK  (687 ms)
410079: wahr = DoS GoldenEye      LLM = ATTACK  (686 ms)
283346: wahr = DoS GoldenEye      LLM = ATTACK  (683 ms)
18421: wahr = DoS Slowhttptest   LLM = ATTACK  (680 ms)
424667: wahr = BENIGN             LL

In [33]:
pd.crosstab(test['label_wahr'], test['label_llm'])

label_llm,ATTACK,BENIGN
label_wahr,,
BENIGN,10,1
DoS GoldenEye,3,0
DoS Hulk,2,0
DoS Slowhttptest,1,0
DoS slowloris,1,2


In [34]:
def ergebnis_art(zeile):
    llm_angriff = zeile['label_llm'] != 'BENIGN'
    if zeile['y'] == 1 and llm_angriff:
        return 'TP (Angriff erkannt)'
    if zeile['y'] == 0 and not llm_angriff:
        return 'TN (korrekt harmlos)'
    if zeile['y'] == 0 and llm_angriff:
        return 'FP (Fehlalarm)'
    return 'FN (Angriff verpasst)'


def begruendung(text):
    try:
        return json.loads(text).get('reason', '')
    except (json.JSONDecodeError, AttributeError):
        return text


uebersicht = test.copy()
uebersicht['ergebnis'] = uebersicht.apply(ergebnis_art, axis=1)
uebersicht['begruendung'] = uebersicht['antwort'].apply(begruendung)

pd.set_option('display.max_colwidth', 200)
uebersicht.sort_values('ergebnis')[['flow_id', 'label_wahr', 'label_llm', 'ergebnis', 'begruendung']]

,flow_id,label_wahr,label_llm,ergebnis,begruendung
9,11984,DoS slowloris,BENIGN,FN (Angriff verpasst),Low packet length variance and standard deviation
10,13320,DoS slowloris,BENIGN,FN (Angriff verpasst),Low packet length variance and standard deviation
17,424667,BENIGN,ATTACK,FP (Fehlalarm),High packet length variance and standard deviation indicate potential malicious activity
13,438286,BENIGN,ATTACK,FP (Fehlalarm),High packet length variance and standard deviation indicate potential malicious activity
12,237195,BENIGN,ATTACK,FP (Fehlalarm),High packet length variance and standard deviation indicate potential malicious activity
11,185749,BENIGN,ATTACK,FP (Fehlalarm),High packet length variance and standard deviation indicate potential malicious activity
8,13817,BENIGN,ATTACK,FP (Fehlalarm),High packet length variance and standard deviation indicate potential malicious activity
0,388132,BENIGN,ATTACK,FP (Fehlalarm),High packet length variance and standard deviation indicate potential malicious activity
6,471952,BENIGN,ATTACK,FP (Fehlalarm),High packet length variance and standard deviation indicate potential malicious activity
2,197415,BENIGN,ATTACK,FP (Fehlalarm),High packet length variance and standard deviation indicate potential malicious activity


In [35]:
few_shot = pd.read_csv(ergebnis_ordner / 'llm_fewshot_beispiele.csv', index_col='Original_Index')
few_shot['Label']

Original_Index
251171          Heartbleed
447713              BENIGN
21152     DoS Slowhttptest
27597        DoS slowloris
229878              BENIGN
21654               BENIGN
313937       DoS GoldenEye
12829               BENIGN
120307            DoS Hulk
Name: Label, dtype: str

In [36]:
def beispiele_text():
    teile = []
    for i, (_, beispiel) in enumerate(few_shot.iterrows(), start=1):
        label = 'ATTACK' if beispiel['y'] == 1 else 'BENIGN'
        teile.append(f'Example {i}:\n{flow_als_text(beispiel)}\nAnswer: {{"label": "{label}"}}')
    return '\n\n'.join(teile)


beispiele = beispiele_text()


def user_prompt_few(zeile):
    return ('Here are labelled example flows:\n\n' + beispiele
            + '\n\nFlow to classify:\n' + flow_als_text(zeile))


print(user_prompt_few(erster_flow)[:1500])

Here are labelled example flows:

Example 1:
- Packet Length Variance: 6.2426e+06
- Packet Length Std: 2498.52
- Bwd Packet Length Std: 2443.18
- Bwd Packet Length Mean: 3928.19
- Average Packet Size: 1643.1
- Bwd Packet Length Max: 13032
- Packet Length Max: 13032
- Subflow Bwd Bytes: 1641
- Packet Length Mean: 1643.1
- Bwd Segment Size Avg: 3928.19
- Total Length of Bwd Packet: 7.88388e+06
- RST Flag Count: 0
- FWD Init Win Bytes: 0
- Fwd Packet Length Max: 66
- Bwd Init Win Bytes: 349
- Fwd Header Length: 89472
- Bwd PSH Flags: 161
- ACK Flag Count: 4803
- Total Bwd packets: 2007
- Bwd Header Length: 64224
Answer: {"label": "ATTACK"}

Example 2:
- Packet Length Variance: 2945.33
- Packet Length Std: 54.2709
- Bwd Packet Length Std: 0
- Bwd Packet Length Mean: 130
- Average Packet Size: 83
- Bwd Packet Length Max: 130
- Packet Length Max: 130
- Subflow Bwd Bytes: 65
- Packet Length Mean: 83
- Bwd Segment Size Avg: 130
- Total Length of Bwd Packet: 260
- RST Flag Count: 0
- FWD Init W

In [37]:
def teste(prompt_funktion, flows):
    ergebnisse = []
    frage_llm(system_prompt, prompt_funktion(flows.iloc[0]))   # Aufwärmen
    for flow_id, zeile in flows.iterrows():
        text, ms, t_in, t_out = frage_llm(system_prompt, prompt_funktion(zeile))
        ergebnisse.append({
            'flow_id': flow_id,
            'label_wahr': zeile['Label'],
            'y': zeile['y'],
            'antwort': text,
            'label_llm': lies_label(text),
            'latenz_ms': ms,
            'tokens_in': t_in,
            'tokens_out': t_out,
        })
    return pd.DataFrame(ergebnisse)


test_few = teste(user_prompt_few, stichprobe.head(n_test))
pd.crosstab(test_few['label_wahr'], test_few['label_llm'])

label_llm,ATTACK,BENIGN
label_wahr,,
BENIGN,4,7
DoS GoldenEye,3,0
DoS Hulk,2,0
DoS Slowhttptest,1,0
DoS slowloris,1,2


In [38]:
ergebnisse_zero = teste(user_prompt, stichprobe)
ergebnisse_zero['model'] = model
ergebnisse_zero['bedingung'] = 'zero_shot'
ergebnisse_zero.to_csv(ergebnis_ordner / 'llm_llama_zero_shot.csv', index=False)

pd.Series(metriken(ergebnisse_zero)).round(3)

FPR           0.950
Recall        0.910
Precision     0.489
F1            0.636
TN            5.000
FP           95.000
FN            9.000
TP           91.000
dtype: float64

In [39]:
ergebnisse_few = teste(user_prompt_few, stichprobe)
ergebnisse_few['modell'] = model
ergebnisse_few['bedingung'] = 'few_shot'
ergebnisse_few.to_csv(ergebnis_ordner / 'llm_llama_few_shot.csv', index=False)

pd.Series(metriken(ergebnisse_few)).round(3)

FPR           0.380
Recall        0.910
Precision     0.705
F1            0.795
TN           62.000
FP           38.000
FN            9.000
TP           91.000
dtype: float64

In [40]:
pd.DataFrame({
    'zero_shot': metriken(ergebnisse_zero),
    'few_shot': metriken(ergebnisse_few),
}).round(3)

,zero_shot,few_shot
FPR,0.950,0.380
Recall,0.910,0.910
Precision,0.489,0.705
F1,0.636,0.795
TN,5.000,62.000
FP,95.000,38.000
FN,9.000,9.000
TP,91.000,91.000


In [41]:
for name, df_erg in [('zero_shot', ergebnisse_zero), ('few_shot', ergebnisse_few)]:
    print(f"{name}: Latenz Median {df_erg['latenz_ms'].median():.0f} ms | "
          f"Tokens im Prompt {df_erg['tokens_in'].mean():.0f}")

zero_shot: Latenz Median 682 ms | Tokens im Prompt 291
few_shot: Latenz Median 649 ms | Tokens im Prompt 2220


In [42]:
import numpy as np

rf_df = pd.DataFrame({
    'y': stichprobe['y'],
    'label_llm': np.where(stichprobe['RF_Vorhersage'] == 1, 'ATTACK', 'BENIGN'),
})
immer_angriff = pd.DataFrame({'y': stichprobe['y'], 'label_llm': 'ATTACK'})

vergleich = pd.DataFrame({
    'Random Forest': metriken(rf_df),
    'Immer ATTACK': metriken(immer_angriff),
    'Llama zero_shot': metriken(ergebnisse_zero),
    'Llama few_shot': metriken(ergebnisse_few),
}).round(3)
vergleich.to_csv(ergebnis_ordner / 'vergleich_rf_llm.csv')
vergleich

,Random Forest,Immer ATTACK,Llama zero_shot,Llama few_shot
FPR,0.0,1.000,0.950,0.380
Recall,1.0,1.000,0.910,0.910
Precision,1.0,0.500,0.489,0.705
F1,1.0,0.667,0.636,0.795
TN,100.0,0.000,5.000,62.000
FP,0.0,100.000,95.000,38.000
FN,0.0,0.000,9.000,9.000
TP,100.0,100.000,91.000,91.000


### Cloud-Modell: OpenAI (ChatGPT)

In [59]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
schluessel = os.getenv('OPENAI_API_KEY', '')
print('Länge:', len(schluessel), '| endet auf:', schluessel[-4:])
client = OpenAI(api_key=schluessel)


Länge: 164 | endet auf: eCcA


In [60]:
modell_openai = 'gpt-6-luna'   # günstigstes GPT-6-Modell


def frage_openai(system, user, versuche=3):
    for versuch in range(versuche):
        try:
            t0 = time.perf_counter()
            r = client.chat.completions.create(
                model=modell_openai,
                messages=[
                    {'role': 'system', 'content': system},
                    {'role': 'user', 'content': user},
                ],
                response_format={'type': 'json_object'},
                reasoning_effort='medium',
                max_completion_tokens=4000,
            )
            dauer_ms = 1000 * (time.perf_counter() - t0)
            return r.choices[0].message.content, dauer_ms, r.usage.prompt_tokens, r.usage.completion_tokens
        except Exception as fehler:
            print(f'Fehler (Versuch {versuch + 1}): {fehler}')
            time.sleep(5 * (versuch + 1))
    return '', None, None, None

In [61]:
text, ms, t_in, t_out = frage_openai(system_prompt, user_prompt(erster_flow))
print('Antwort:', text)
print(f'Dauer: {ms:.0f} ms | Tokens: {t_in} gelesen, {t_out} geschrieben')

preis_input, preis_output = 0.10, 0.50   
kosten = (t_in * preis_input + t_out * preis_output) / 1e6
print(f'Kosten für diesen Flow: {kosten:.5f} $ | geschätzt für 400 Aufrufe: {kosten * 400:.2f} $')

Antwort: {"label":"BENIGN","reason":"The flow is a small, low-volume exchange with no clear indicators of attack activity."}
Dauer: 5083 ms | Tokens: 277 gelesen, 224 geschrieben
Kosten für diesen Flow: 0.00014 $ | geschätzt für 400 Aufrufe: 0.06 $


In [62]:
def teste(prompt_funktion, flows, frage=frage_llm):
    ergebnisse = []
    frage(system_prompt, prompt_funktion(flows.iloc[0]))   # Aufwärmen
    for flow_id, zeile in flows.iterrows():
        text, ms, t_in, t_out = frage(system_prompt, prompt_funktion(zeile))
        ergebnisse.append({
            'flow_id': flow_id,
            'label_wahr': zeile['Label'],
            'y': zeile['y'],
            'antwort': text,
            'label_llm': lies_label(text),
            'latenz_ms': ms,
            'tokens_in': t_in,
            'tokens_out': t_out,
        })
    return pd.DataFrame(ergebnisse)

In [63]:
ergebnisse_gpt_zero = teste(user_prompt, stichprobe, frage=frage_openai)
ergebnisse_gpt_zero['model'] = modell_openai
ergebnisse_gpt_zero['bedingung'] = 'zero_shot'
ergebnisse_gpt_zero.to_csv(ergebnis_ordner / 'llm_openai_zero_shot.csv', index=False)

ergebnisse_gpt_few = teste(user_prompt_few, stichprobe, frage=frage_openai)
ergebnisse_gpt_few['model'] = modell_openai
ergebnisse_gpt_few['bedingung'] = 'few_shot'
ergebnisse_gpt_few.to_csv(ergebnis_ordner / 'llm_openai_few_shot.csv', index=False)

tokens_in = ergebnisse_gpt_zero['tokens_in'].sum() + ergebnisse_gpt_few['tokens_in'].sum()
tokens_out = ergebnisse_gpt_zero['tokens_out'].sum() + ergebnisse_gpt_few['tokens_out'].sum()
print(f'Tokens: {tokens_in} gelesen, {tokens_out} geschrieben | '
      f'Kosten ca. {(tokens_in * preis_input + tokens_out * preis_output) / 1e6:.2f} $')

Tokens: 499180 gelesen, 88126 geschrieben | Kosten ca. 0.09 $


In [64]:
vergleich = pd.DataFrame({
    'Random Forest': metriken(rf_df),
    'Immer ATTACK': metriken(immer_angriff),
    'Llama zero_shot': metriken(ergebnisse_zero),
    'Llama few_shot': metriken(ergebnisse_few),
    'GPT-6 Luna zero_shot': metriken(ergebnisse_gpt_zero),
    'GPT-6 Luna few_shot': metriken(ergebnisse_gpt_few),
}).round(3)
vergleich.to_csv(ergebnis_ordner / 'vergleich_rf_llm.csv')
vergleich

,Random Forest,Immer ATTACK,Llama zero_shot,Llama few_shot,GPT-6 Luna zero_shot,GPT-6 Luna few_shot
FPR,0.0,1.000,0.950,0.380,0.120,0.420
Recall,1.0,1.000,0.910,0.910,0.640,0.990
Precision,1.0,0.500,0.489,0.705,0.842,0.702
F1,1.0,0.667,0.636,0.795,0.727,0.822
TN,100.0,0.000,5.000,62.000,88.000,58.000
FP,0.0,100.000,95.000,38.000,12.000,42.000
FN,0.0,0.000,9.000,9.000,36.000,1.000
TP,100.0,100.000,91.000,91.000,64.000,99.000


In [66]:
for name, erg in [('Llama zero_shot', ergebnisse_zero), ('Llama few_shot', ergebnisse_few),
                  ('GPT-6 Luna zero_shot', ergebnisse_gpt_zero), ('GPT-6 Luna few_shot', ergebnisse_gpt_few)]:
    print(f"{name:22s} Latenz Median {erg['latenz_ms'].median():7.0f} ms | "
          f"Tokens geschrieben {erg['tokens_out'].mean():6.0f}")

Llama zero_shot        Latenz Median     682 ms | Tokens geschrieben     28
Llama few_shot         Latenz Median     649 ms | Tokens geschrieben     26
GPT-6 Luna zero_shot   Latenz Median    4888 ms | Tokens geschrieben    339
GPT-6 Luna few_shot    Latenz Median    1922 ms | Tokens geschrieben    101
